# Упражнение: Лаговые признаки без утечки и walk-forward проверка

## Задание

В `input.csv` — дневные продажи магазина за 140 дней (`date, sales`, с 2024-01-01).
У ряда есть тренд, недельная сезонность и «инерция» (вчерашний всплеск частично продолжается сегодня) —
это авторегрессионная часть, о которой рассказывает лекция про ARIMA. Постройте авторегрессию
на лаговых признаках и честно проверьте её **скользящим началом прогноза** (walk-forward).

1. Признаки для дня t (строго из прошлого):
   - `lag_1` = sales(t − 1), `lag_7` = sales(t − 7);
   - `roll_mean_7` = среднее sales(t − 7), …, sales(t − 1) — **без** самого sales(t).
2. Тест — последние 28 дней, 4 фолда по 7 дней подряд. Для фолда k:
   - обучающая выборка — все дни **до** начала фолда, у которых все три признака определены
     (первые 7 дней ряда выпадают);
   - модель — `LinearRegression()` на `lag_1, lag_7, roll_mean_7` → `sales`;
   - прогноз на шаг вперёд для 7 дней фолда (признаки берутся из фактических прошлых значений).
3. Бейзлайн — **сезонный наивный** прогноз: ŷ(t) = sales(t − 7).
4. Для каждого фолда посчитайте MAE модели и MAE бейзлайна.

## Подсказки

- Случайный train/test здесь запрещён: обучение — только на днях раньше фолда, и модель
  переобучается заново на каждом фолде (обучающая выборка растёт на 7 дней).
- Скользящее среднее «из прошлого» — это `shift(1)` **перед** `rolling(7)`. Если написать
  `rolling(7).mean()` без сдвига, в признак попадёт сам sales(t) — классическая утечка:
  в момент прогноза такой признак посчитать нельзя.
- Признаки: `df['sales'].shift(1)`, `.shift(7)`, `.shift(1).rolling(7).mean()`; позиция начала фолда
  k = 0..3 — `len(df) - 28 + 7*k`; train — строки с позицией меньше начала и без NaN;
  сезонный наивный прогноз — это просто колонка `lag_7` тестовых строк.

## Ожидаемый формат выходного файла

CSV с колонками `fold, test_start, train_size, mae_model, mae_snaive` — 4 строки (fold = 1..4 по порядку);
`test_start` — дата первого дня фолда в формате `YYYY-MM-DD`, `train_size` — число обучающих строк.
MAE округлить до 4 знаков.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/time_series
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.